# Faster R-CNN — CoralSense Colab Training

Use the reviewed Hard Coral / Soft Coral / quadrat_area dataset version. Split by survey/session before train-only augmentation. Run from a fresh GPU runtime.

Configure the version in Step 2. Check validation results to select models; reserve test for final evaluation. Checkpoints and metadata persist to Drive by default. Training budgets/pretraining differ by architecture and must be reported.

Validation is the default evaluation split. Before a long run, use one epoch as a smoke test, inspect boxes and losses, then start a fresh full experiment. Old five-class checkpoints cannot be used in the new comparison.


## Step 1: Install Dependencies

In [ ]:
# Use Colab's installed Torch/CUDA pair; do not replace it with cu118 wheels.
!pip install -q roboflow pyyaml torchmetrics pycocotools tqdm

## Step 2: Download Dataset from Roboflow

In [ ]:
MODEL_NAME="faster_rcnn"
from pathlib import Path
from getpass import getpass
from datetime import datetime
import os, json, random, hashlib, importlib.metadata
import numpy as np
import torch
from roboflow import Roboflow

# Fill these from the newly generated Roboflow version. Never paste a key here.
WORKSPACE = ""
PROJECT = ""
DATASET_VERSION = None
EXPECTED_CLASSES = {"Hard Coral", "Soft Coral", "quadrat_area"}
SEED = 42
USE_DRIVE = True
EVAL_SPLIT = "valid"  # Use test only for the final, frozen evaluation.
assert WORKSPACE and PROJECT and isinstance(DATASET_VERSION, int), "Enter workspace, project and version first."
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = Path('/content/drive/MyDrive/CoralSense')
else:
    OUTPUT_ROOT = Path.cwd() / 'coralsense_outputs'
CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUTPUT_ROOT / 'runs' / (datetime.now().strftime('%Y%m%d-%H%M%S-%f') + '-' + MODEL_NAME)
RUN_DIR.mkdir(parents=True)
ROBOFLOW_API_KEY = getpass("Roboflow API key: ")
dataset = Roboflow(api_key=ROBOFLOW_API_KEY).workspace(WORKSPACE).project(PROJECT).version(DATASET_VERSION).download('yolov8')
del ROBOFLOW_API_KEY
dataset_path = Path(dataset.location)

## Step 3: Verify Dataset Structure

In [ ]:
import yaml
from PIL import Image, ImageOps
import torchvision.transforms.functional as TF

with (dataset_path / 'data.yaml').open() as handle:
    data = yaml.safe_load(handle)
names = data['names']
class_names = [names[k] for k in sorted(names, key=int)] if isinstance(names, dict) else list(names)
assert len(class_names) == 3 and set(class_names) == EXPECTED_CLASSES, f"Unexpected classes: {class_names}"
data['names'] = class_names
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}
def image_paths(directory):
    return sorted(p for p in Path(directory).iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)

def find_split(dp, split_names):
    for name in split_names:
        aliases = ['valid', 'val'] if name in ('valid', 'val') else [name]
        for split in aliases:
            for images, labels in [(dp / split / 'images', dp / split / 'labels'),
                                   (dp / 'images' / split, dp / 'labels' / split)]:
                if images.is_dir() and labels.is_dir() and image_paths(images): return images, labels
    raise FileNotFoundError(f"Missing/empty split: {split_names}. No silent evaluation fallback.")

def read_annotations(path, label_dir, width, height, offset=0):
    boxes, labels = [], []
    label_path = Path(label_dir) / (Path(path).stem + '.txt')
    if not label_path.exists():
        raise FileNotFoundError(f"Missing label: {label_path.name}. Use an empty file only for reviewed negative images.")
    for line in label_path.read_text().splitlines():
        if not line.strip(): continue
        fields = line.split()
        if len(fields) != 5: raise ValueError(f"Expected detection boxes, not polygons: {label_path.name}")
        cid = int(fields[0]); cx, cy, w, h = map(float, fields[1:])
        if not (0 <= cid < len(class_names) and all(np.isfinite([cx,cy,w,h])) and
                0 <= cx <= 1 and 0 <= cy <= 1 and 0 < w <= 1 and 0 < h <= 1):
            raise ValueError(f"Invalid class/box in {label_path.name}")
        box = [max(0., (cx-w/2)*width), max(0., (cy-h/2)*height),
               min(float(width), (cx+w/2)*width), min(float(height), (cy+h/2)*height)]
        if box[2] <= box[0] or box[3] <= box[1]: raise ValueError(f"Degenerate box: {label_path.name}")
        boxes.append(box); labels.append(cid + offset)
    return {'boxes': torch.tensor(boxes, dtype=torch.float32).reshape(-1,4),
            'labels': torch.tensor(labels, dtype=torch.int64)}

split_dirs = {s: find_split(dataset_path, [s]) for s in ('train','valid','test')}
manifest = {}; seen_hashes = {}; fingerprint = hashlib.sha256()
for split, (images_dir, labels_dir) in split_dirs.items():
    counts = np.zeros(len(class_names), dtype=int); rows = []
    for path in image_paths(images_dir):
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if digest in seen_hashes and seen_hashes[digest] != split:
            raise ValueError('Identical image found across splits: ' + path.name)
        seen_hashes[digest] = split
        with Image.open(path) as img: target = read_annotations(path, labels_dir, *img.size)
        for cid in target['labels'].tolist(): counts[cid] += 1
        label_hash = hashlib.sha256((labels_dir / (path.stem+'.txt')).read_bytes()).hexdigest()
        rows.append({'image_sha256': digest, 'label_sha256': label_hash})
    if any(counts == 0): raise ValueError(f"{split} has no examples of: {[class_names[i] for i in np.where(counts==0)[0]]}")
    # Filenames differ across exports; content-based fingerprints remain comparable.
    rows.sort(key=lambda r: (r['image_sha256'],r['label_sha256']))
    manifest[split] = {'images':len(rows),'instances':counts.tolist(),'content':rows}
    print(split, 'images:',len(rows), 'instances:',dict(zip(class_names,counts.tolist())))
fingerprint.update(json.dumps({'classes':class_names,'splits':manifest},sort_keys=True).encode())
DATASET_FINGERPRINT = fingerprint.hexdigest()
META = {'workspace':WORKSPACE,'project':PROJECT,'version':DATASET_VERSION,
        'class_names':class_names,'dataset_fingerprint':DATASET_FINGERPRINT,'seed':SEED,
        'model':MODEL_NAME,'hardware':torch.cuda.get_device_name(0) if device.type=='cuda' else 'cpu',
        'augmentation':'Roboflow version transformations only; additional random training augmentation disabled'}
(RUN_DIR/'dataset_manifest.json').write_text(json.dumps(manifest,indent=2))
(RUN_DIR/'run_metadata.json').write_text(json.dumps(META,indent=2))
versions = {}
for name in ('torch','torchvision','roboflow','numpy','Pillow','ultralytics','torchmetrics','effdet','timm','pycocotools'):
    try: versions[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: pass
(RUN_DIR/'package_versions.json').write_text(json.dumps(versions,indent=2))
print('Classes:',class_names,'Output:',RUN_DIR)
print('Exact duplicates checked. Related photos must also have been grouped by survey/session before export.')

def letterbox(img, size):
    w,h=img.size; ratio=min(size/w,size/h)
    nw,nh=max(1,round(w*ratio)),max(1,round(h*ratio))
    left,top=(size-nw)//2,(size-nh)//2
    canvas=Image.new('RGB',(size,size),(114,114,114))
    canvas.paste(img.resize((nw,nh),Image.Resampling.BILINEAR),(left,top))
    return canvas,(nw/w,nh/h,left,top)

## Step 4: Prepare Data for Faster R-CNN

In [ ]:
from torch.utils.data import Dataset, DataLoader
class CoralDataset(Dataset):
    def __init__(self, images, labels, efficientdet=False):
        self.paths=image_paths(images); self.labels=labels; self.efficientdet=efficientdet
    def __len__(self): return len(self.paths)
    def __getitem__(self,index):
        path=self.paths[index]
        with Image.open(path) as original: img=original.convert('RGB')
        target=read_annotations(path,self.labels,*img.size,offset=1)
        if self.efficientdet:
            img,(sx,sy,left,top)=letterbox(img,IMG_SIZE)
            boxes=target['boxes'].clone()
            boxes[:,[0,2]]=boxes[:,[0,2]]*sx+left
            boxes[:,[1,3]]=boxes[:,[1,3]]*sy+top
            tensor=TF.normalize(TF.to_tensor(img),[.485,.456,.406],[.229,.224,.225])
            target={'bbox':boxes[:,[1,0,3,2]],'cls':target['labels']}
        else:
            tensor=TF.to_tensor(img)
        return tensor,target
def collate(batch): return tuple(zip(*batch))

## Step 5: Train Faster R-CNN Model

In [ ]:
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm.auto import tqdm
num_classes=len(class_names)+1
model=fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1,
                             box_score_thresh=.001,box_detections_per_img=100)
model.roi_heads.box_predictor=FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features,num_classes)
model.to(device)
train_loader=DataLoader(CoralDataset(*split_dirs['train']),batch_size=4,shuffle=True,collate_fn=collate,num_workers=2)
valid_loader=DataLoader(CoralDataset(*split_dirs['valid']),batch_size=1,collate_fn=collate,num_workers=2)
optimizer=torch.optim.SGD(model.parameters(),lr=.005,momentum=.9,weight_decay=.0005)
scheduler=torch.optim.lr_scheduler.StepLR(optimizer,step_size=8,gamma=.1)
num_epochs=20
def evaluate(loader):
    model.eval(); metric=MeanAveragePrecision(box_format='xyxy',class_metrics=True)
    with torch.inference_mode():
        for images,targets in loader:
            predictions=model([x.to(device) for x in images])
            metric.update([{k:v.cpu() for k,v in p.items() if k in ('boxes','scores','labels')} for p in predictions],list(targets))
    return metric.compute()
META['training']={'epochs':num_epochs,'batch':4,'optimizer':'SGD','pretraining':'COCO detection','resize':'torchvision native transform'}
(RUN_DIR/'run_metadata.json').write_text(json.dumps(META,indent=2))
best_map=-1.; logs=[]
for epoch in range(num_epochs):
    model.train(); losses=[]
    for images,targets in tqdm(train_loader,desc=f'Epoch {epoch+1}'):
        loss=sum(model([x.to(device) for x in images],[{k:v.to(device) for k,v in t.items()} for t in targets]).values())
        if not torch.isfinite(loss): raise RuntimeError('Non-finite training loss')
        optimizer.zero_grad();loss.backward();optimizer.step();losses.append(loss.item())
    scheduler.step(); val=evaluate(valid_loader); score=float(val['map'])
    checkpoint={'model_state_dict':model.state_dict(),'num_classes':num_classes,'class_names':class_names,
                'epoch':epoch+1,'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),'metadata':META}
    torch.save(checkpoint,RUN_DIR/'last.pt')
    if score>best_map:
        best_map=score;torch.save(checkpoint,RUN_DIR/'best.pt')
    logs.append({'epoch':epoch+1,'loss':float(np.mean(losses)),'validation_map':score})
    (RUN_DIR/'training_log.json').write_text(json.dumps(logs,indent=2))
    print(logs[-1])
# Load only your own trusted checkpoint; optimizer checkpoints need weights_only=False.
model.load_state_dict(torch.load(RUN_DIR/'best.pt',map_location=device,weights_only=False)['model_state_dict'])
model.eval()

## Step 6: Evaluate on Validation Set

In [ ]:
IS_EDET=False
# Validation is the default. Keep test untouched until model/settings selection is final.
eval_img_dir,eval_lbl_dir=split_dirs[EVAL_SPLIT]
eval_loader=DataLoader(CoralDataset(eval_img_dir,eval_lbl_dir,efficientdet=IS_EDET),
                       batch_size=1,collate_fn=collate,num_workers=2)
res=evaluate(eval_loader)
print('Evaluation split:',EVAL_SPLIT,'mAP50:',float(res['map_50']),'mAP50-95:',float(res['map']))
for cid,ap in zip(res['classes'].tolist(),res['map_per_class'].reshape(-1).tolist()):
    print(class_names[cid-1],ap)
(RUN_DIR/'evaluation.json').write_text(json.dumps({'split':EVAL_SPLIT,'map50':float(res['map_50']),
    'map':float(res['map']),'classes':res['classes'].tolist(),'per_class_ap':res['map_per_class'].reshape(-1).tolist()},indent=2))

## Step 7: Test on Sample Images

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
for path in image_paths(eval_img_dir)[:3]:
    with Image.open(path) as original:img=original.convert('RGB')
    if IS_EDET:
        img,_=letterbox(img,IMG_SIZE)
        tensor=TF.normalize(TF.to_tensor(img),[.485,.456,.406],[.229,.224,.225]).unsqueeze(0).to(device)
        with torch.inference_mode():rows=predict_bench(tensor)[0].cpu()
        boxes,scores,labels=rows[:,:4],rows[:,4],rows[:,5].long()
    else:
        with torch.inference_mode():p=model([TF.to_tensor(img).to(device)])[0]
        boxes,scores,labels=p['boxes'].cpu(),p['scores'].cpu(),p['labels'].cpu()
    fig,ax=plt.subplots(figsize=(8,8));ax.imshow(img);ax.set_title(path.name)
    for box,score,label in zip(boxes,scores,labels):
        if score<.3:continue
        x1,y1,x2,y2=box.tolist()
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor='cyan'))
        ax.text(x1,y1,f'{class_names[int(label)-1]} {float(score):.2f}',color='cyan')
    plt.show()

## Step 8: Save and Export Model

In [ ]:
CHECKPOINT_NAME="faster_rcnn_coral_best.pt"
import shutil
model_path=CHECKPOINT_DIR / CHECKPOINT_NAME
shutil.copy2(RUN_DIR/'best.pt',model_path)
print('Best checkpoint:',model_path)
print('Latest checkpoint (model, optimizer, scheduler, epoch):',RUN_DIR/'last.pt')
print('Keep the run directory for reproducibility; use matching versions to resume.')

## Step 9: Download Trained Model

In [ ]:
# Files are already persisted on Drive when USE_DRIVE=True.
import shutil
archive=shutil.make_archive('/content/'+MODEL_NAME+'_run','zip',RUN_DIR)
from google.colab import files
files.download(archive)
print('Published checkpoint:',model_path)